# Demo 13 — LM with mixed predictors and VIF

This demo uses the `mtcars` dataset (R base; Henderson & Velleman 1981),
performance figures for 32 car models. Fuel economy (`mpg`) is modelled from
engine power (`hp`) and weight (`wt`) — two strongly correlated predictors —
together with the number of cylinders (`cyl`) as a categorical factor.

The fitted model combines the categorical and numeric predictors:

    mpg ~ cyl + hp + wt

Because `options.x` contains numeric predictors, kbstatpy automatically reports
their Variance Inflation Factors (thresholds: < 5 OK, 5–10 concerning, > 10
severe), flagging the collinearity between power and weight that would otherwise
inflate standard errors and destabilise the coefficients. A correlation scatter
grid with VIF on the diagonal visualises it.

## Setup

In [ ]:
import os
try:
    notebook_dir = os.path.dirname(os.path.abspath(__vsc_ipynb_file__))
except NameError:
    notebook_dir = os.getcwd()  # JupyterLab sets CWD to the notebook directory
os.chdir(notebook_dir)

import matplotlib.pyplot as plt
from kbstatpy import Kbstat, KbstatOptions

## Options

`covariate = 'hp, wt'` adds continuous covariates to the model without including them in the data plot. `correlation = 'hp, wt'` triggers VIF computation and a pairwise scatter plot of the numeric predictors.

In [ ]:
options = KbstatOptions()
options.in_file     = os.path.join(notebook_dir, '../data/mtcars.csv')
options.out_dir     = os.path.join(notebook_dir, 'results/demo_13_lm_vif')
options.y           = 'mpg'
options.y_units     = 'mpg'
options.x           = 'cyl'
options.rename      = 'mpg -> Consumption; cyl -> Cylinders; hp -> Horsepower; wt -> Weight'
options.x_order     = 'cyl: 4, 6, 8'
options.covariate   = 'hp, wt'
options.correlation = 'hp, wt'

## Model fitting

`fit()` estimates the model parameters via restricted maximum likelihood (REML).

In [ ]:
kb = Kbstat(options)
kb.fit()
print(f'Formula : {kb._build_formula()}')
print(f'AIC     : {kb.AIC:.3f}')
print(f'BIC     : {kb.BIC:.3f}')
print(f'logLik  : {kb.logLik:.3f}')

## ANOVA table (Type III)

In [ ]:
kb.anova()
kb.anova_table

## Post-hoc pairwise comparisons

Pairwise comparisons across cylinder counts (4, 6, 8).

In [ ]:
kb.posthoc()
kb.posthoc_table

## Data plot

Violin + jitter scatter with EMM and 95 % CI overlaid.

> An interactive version with hover tooltips is saved to `results/` as an HTML file.

In [ ]:
kb.plot_data()
plt.show()

## Diagnostic plots

Six panels checking model assumptions.

See [STATISTICAL_NOTES.md](../../STATISTICAL_NOTES.md) for interpretation guidance.

In [ ]:
kb.plot_diagnostics()
plt.show()

## Save results

Writes all output files to `out_dir`.

In [ ]:
kb.save()

## Interpretation

- VIF values above 5 indicate multicollinearity — hp and wt in mtcars are highly correlated, so VIF is expected to be elevated.
- Despite multicollinearity, the model can still estimate effects if the question is predictive rather than causal.
- The data plot shows cylinder group means with hp and wt held at their grand means (covariate-adjusted EMMs).